In [1]:
%pip install collections

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement collections (from versions: none)
ERROR: No matching distribution found for collections


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque


class DQN(nn.Module):
    def __init__(self, states, actions):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(states, 64),
            nn.ReLU(),
            nn.Linear(64, actions)
        )

    def forward(self, x):
        return self.model(x)


class PERBuffer:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
        self.priorities = deque(maxlen=capacity)

    def add(self, data, error):
        self.memory.append(data)
        self.priorities.append(abs(error) + 1e-5)

    def sample(self, size):
        probabilities = np.array(self.priorities)
        probabilities /= probabilities.sum()

        indices = np.random.choice(
            len(self.memory),
            size=size,
            p=probabilities
        )

        return [self.memory[i] for i in indices], indices

    def update(self, indices, errors):
        for index, error in zip(indices, errors):
            self.priorities[index] = abs(error) + 1e-5


env = gym.make("CartPole-v1")

policy_net = DQN(
    env.observation_space.shape[0],
    env.action_space.n
)

target_net = DQN(
    env.observation_space.shape[0],
    env.action_space.n
)

target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
memory = PERBuffer(1000)

state, _ = env.reset()
action = env.action_space.sample()

next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

state_tensor = torch.tensor(state, dtype=torch.float32)
next_state_tensor = torch.tensor(next_state, dtype=torch.float32)

with torch.no_grad():
    current_q = policy_net(state_tensor)[action]
    next_q = target_net(next_state_tensor).max()
    target_q = reward + 0.99 * next_q * (not done)
    error = target_q - current_q

memory.add(
    (state, action, reward, next_state, done),
    error.item()
)

batch, indices = memory.sample(1)
state, action, reward, next_state, done = batch[0]

predicted_q = policy_net(
    torch.tensor(state, dtype=torch.float32)
)[action]

with torch.no_grad():
    target_q = reward + 0.99 * target_net(
        torch.tensor(next_state, dtype=torch.float32)
    ).max() * (not done)

loss = nn.functional.mse_loss(predicted_q, target_q)

optimizer.zero_grad()
loss.backward()
optimizer.step()

memory.update(
    indices,
    [(target_q - predicted_q).item()]
)

env.close()

print("DQN with PER updated successfully!")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\mahab\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.